# Asynchronous Inference with Ray Serve

This notebook covers how to build asynchronous inference APIs using Ray Serve

<div class="alert alert-info">
<strong>Roadmap for this notebook</strong>
<ol>
  <li>Why Asynchronous Inference? </li>
  <li>Architecture Overview </li>
  <li>Prerequisites </li>
  <li>Basic Example </li>
  <li>Configuration Reference </li>
  <li>Concurrency and Scaling </li>
  <li>Reliability and Error Handling </li>
  <li>Monitoring </li>
  <li>Task Status Lifecycle </li>
  <li>Advanced Patterns </li>
  <li>Troubleshooting </li>
</ol>
</div>

**Imports**

In [ ]:
import json
import requests
import redis
from ray import serve
from ray.serve.task_consumer import task_consumer, task_handler, instantiate_adapter_from_config
from ray.serve.handle import DeploymentHandle
from ray.serve.config import AutoscalingConfig
from fastapi import FastAPI
from pydantic import BaseModel

## Why Asynchronous Inference?

Synchronous APIs block until processing completes. This is problematic for long-running tasks like video processing or document analysis.

Asynchronous inference decouples request submission from result retrieval as shown in the diagram below.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/sync_vs_async_inference.png" alt="Ray Serve Logo" width="700">

**Benefits:**
- Clients aren't blocked during long operations
- Built-in retry and failure handling (requests are retried until successful)
- Natural backpressure via queue—workers process at steady pace regardless of intake


## Architecture Overview

The following diagram shows the components involved in an asynchronous inference request:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/async_inference_architecture.png" alt="Ray Serve Logo" width="900">


**Components:**
- **Load Balancer**: External LB (ALB, nginx, Traefik) distributes traffic to Ray cluster
- **Ray Serve Proxy**: Routes requests to deployment replicas, handles health checks
- **Ingress Deployment**: HTTP API that enqueues tasks and retrieves status
- **Broker**: Message queue holding tasks to be processed (e.g RabbitMQ, Redis)
- **Backend**: Storage for task results and status (e.g Redis)
- **Task Consumer**: Ray Serve deployment running Celery worker threads (pulls from broker)



### Installing Dependencies

The `ray[serve-async-inference]` contains additional dependencies for async inference, primarily the celery pacakge.

If testing this out with Redis as the broker+backend, install the following:

```bash
pip install "ray[serve-async-inference]" "celery[redis]"
```

Or in a Dockerfile:
```dockerfile
FROM anyscale/ray:2.53.0-slim-py312

RUN pip install --no-cache-dir \
    "ray[serve-async-inference]==2.53.0" \
    "celery[redis]==5.5.3"

# Install Redis server (for local development)
RUN sudo apt-get update -qq \
    && sudo apt-get install --no-install-recommends -y redis-server redis-tools \
    && sudo rm -rf /var/lib/apt/lists/*
```

### Start Redis

For local development:

In [ ]:
# Uncomment to get server running
# !redis-server --port 6399 --daemonize yes

For production, use a managed Redis service (AWS ElastiCache, Redis Cloud, etc.)

In [ ]:
# Let's check that Redis server is running
!redis-cli -p 6399 ping

## Basic Example

Let's go over a basic example of how to set up an async inference API using Ray Serve.

### Step 1: Define Configuration

First, we need to define the configuration for the async inference API. 

In [ ]:
from ray.serve.schema import CeleryAdapterConfig, TaskProcessorConfig

REDIS_URL = "redis://127.0.0.1:6399/0"

task_processor_config = TaskProcessorConfig(
    queue_name="inference_queue",
    max_retries=3, # number times to retry a task
    adapter_config=CeleryAdapterConfig(
        broker_url=REDIS_URL,  # point to Broker implementation
        backend_url=REDIS_URL, # point to Backend implementation
    )
)

In case of failure, we can use a dead letter queue to store the task, otherwise tasks will be discarded and we won't be able to inspect it or replay it once the issue is fixed.

In [ ]:
task_processor_config = TaskProcessorConfig(
    queue_name="inference_queue",
    max_retries=3, # number times to retry a task
    adapter_config=CeleryAdapterConfig(
        broker_url=REDIS_URL,  # point to Broker implementation
        backend_url=REDIS_URL, # point to Backend implementation
    ),
    failed_task_queue_name="inference_failed",
)

You can also add an unprocessable task queue to store tasks that are not processable (e.g)

### Step 2: Create the Task Consumer

Let's create a simple task consumer that will process the tasks.

In [ ]:
@serve.deployment(
    num_replicas=2,
    max_ongoing_requests=5,  # Sets Celery worker thread count per replica
    ray_actor_options={"resources": {"anyscale/node-group:head": 0.0001}}
)
@task_consumer(task_processor_config=task_processor_config)
class ThreadBasedTaskProcessor:
    """Task consumer using Celery's thread pool."""
    def __init__(self, gpu_model: DeploymentHandle):
        # Get handles to downstream GPU deployments for heavy model inference
        self.gpu_model = gpu_model
    
    @task_handler(name="run_inference")
    def run_inference(self, data: dict) -> dict:
        """
        Orchestrate inference by delegating to GPU deployment.
        
        The thread handles I/O and orchestration while GPU work
        runs on dedicated GPU replicas.
        """
        if data.get("fail", False):
            raise ValueError(f"Failed to process {data=}")

        # Delegate heavy lifting to GPU deployment
        prediction = self.gpu_model.predict.remote(data).result()
        
        return {
            "input": data,
            "prediction": prediction,
            "status": "completed"
        }

Here is the definition of a naive GPU-based deployment

In [ ]:
@serve.deployment(
    autoscaling_config=AutoscalingConfig(
        min_replicas=1,
        max_replicas=2,
        target_ongoing_requests=204, # Autoscale based on target of ongoing requests per replica
    ),
    max_ongoing_requests=256,
    ray_actor_options={"num_gpus": 1},  # Each replica gets 1 GPU
)
class GPUModelDeployment:
    def __init__(self):
        # Load model onto GPU
        self.model = self._load_model()

    def _load_model(self):
        # Load your actual model here
        return "model_v1"

    @serve.batch(max_batch_size=128, batch_wait_timeout_s=0.1)
    async def predict(self, data: list[dict]) -> list:
        """Run GPU inference."""
        return [0.1 for elem in data]

### Step 3: Create the HTTP Ingress

Let's create the HTTP ingress deployment that will enqueue tasks and retrieve status.

In [ ]:
app = FastAPI()


class InferenceRequest(BaseModel):
    data: dict


class TaskResponse(BaseModel):
    task_id: str
    status: str


class TaskStatusResponse(BaseModel):
    task_id: str
    status: str
    result: dict | None = None


@serve.deployment(ray_actor_options={"resources": {"anyscale/node-group:head": 0.0001}})
@serve.ingress(app)
class AsyncInferenceAPI:
    def __init__(self, consumer_handler: DeploymentHandle):
        self.adapter = instantiate_adapter_from_config(task_processor_config)

    @app.post("/submit", response_model=TaskResponse)
    async def submit_task(self, request: InferenceRequest):
        """Submit an inference task and return immediately with task ID."""
        result = self.adapter.enqueue_task_sync(
            task_name="run_inference", kwargs={"data": request.data}
        )
        return TaskResponse(task_id=result.id, status=result.status)

    @app.get("/status/{task_id}", response_model=TaskStatusResponse)
    async def get_status(self, task_id: str):
        """Check the status of a submitted task."""
        result = self.adapter.get_task_status_sync(task_id)
        task_result = result.result
        if isinstance(task_result, Exception):
            task_result = {"error": str(task_result)}
        return TaskStatusResponse(
            task_id=result.id, status=result.status, result=task_result
        )

    @app.delete("/cancel/{task_id}")
    async def cancel_task(self, task_id: str):
        """Cancel a pending task."""
        self.adapter.cancel_task_sync(task_id)
        return {"task_id": task_id, "cancelled": True}

### Step 4: Deploy the Application

Let's proceed to deploy the application

In [ ]:
def build_app():
    """Build and configure the Ray Serve application."""
    # Deploy background worker
    consumer = ThreadBasedTaskProcessor.bind(gpu_model=GPUModelDeployment.bind())

    # Deploy HTTP API
    api = AsyncInferenceAPI.bind(consumer)

    return api


# Entry point for Ray Serve
app = build_app()

We can visualize the celery tasks using flower

In [ ]:
# Uncomment and run in your terminal
# !celery --broker=redis://127.0.0.1:6399/0 flower --port=5556

Let's run the application

In [ ]:
serve.run(app, blocking=False)

Let's send a request to the application and get back the task ID

In [ ]:
response = requests.post("http://localhost:8000/submit", json={"data": {"text": "Hello World"}})
response.status_code, response.json()

In [ ]:
task_id = response.json()["task_id"]

Let's now poll the status of the task

In [ ]:
response = requests.get(f"http://localhost:8000/status/{task_id}")
response.json()

Let's verify the failed task was sent to the dead letter queue

In [ ]:
response = requests.post("http://localhost:8000/submit", json={"data": {"fail": "bad"}})
response.status_code, response.json()

We should get back a failed status and a result with the error message

In [ ]:
task_id = response.json()["task_id"]
response = requests.get(f"http://localhost:8000/status/{task_id}")
response.json()

We can inspect the dead letter queue to view the failed task

In [ ]:
r = redis.Redis(host='localhost', port=6399, db=0)

# Check queue length
queue_len = r.llen("inference_failed")
print(f"DLQ has {queue_len} messages")

# Read messages without removing (peek)
messages = r.lrange("inference_failed", 0, -1)
for i, msg in enumerate(messages):
    decoded = json.loads(msg)
    print(f"\n--- Message {i} ---")
    print(json.dumps(decoded, indent=2))

### Handling Soft Timeouts

Catch `SoftTimeLimitExceeded` to perform cleanup before the hard timeout kills the task:

In [58]:
task_processor_config.adapter_config.task_custom_config = (
    {
        "time_limit": 300,  # Hard timeout limit
        "soft_time_limit": 270,  # soft timeout limit
    },
)

In [59]:
from celery.exceptions import SoftTimeLimitExceeded

@serve.deployment
@task_consumer(task_processor_config=task_processor_config)
class ProcessorWithTimeout:
    @task_handler(name="run_inference")
    def run_inference(self, data: dict) -> dict:
        try:
            result = self.do_long_processing(data)
            return result
        except SoftTimeLimitExceeded:
            # Cleanup and save partial progress before hard timeout
            self.save_partial_result(data)
            raise  # Re-raise to mark task as failed
    
    def do_long_processing(self, data):
        # Long-running work here
        pass
    
    def save_partial_result(self, data):
        # Save progress so work isn't completely lost
        pass

### Per-Task Timeout Override

You can control the timeout for specific tasks by overriding the default timeout when enqueuing specific tasks:

```python
# Use longer timeout for this specific task
result = adapter.enqueue_task_sync(
    task_name="run_inference",
    kwargs={"data": large_data},
    time_limit=600,        # 10 min hard timeout (overrides default)
    soft_time_limit=550,   # 9 min soft timeout
)

# Use shorter timeout for quick tasks
result = adapter.enqueue_task_sync(
    task_name="quick_validation",
    kwargs={"data": small_data},
    time_limit=30,         # 30 sec hard timeout
    soft_time_limit=25,
)
```

## Best Practices

1. **Make handlers idempotent** — Tasks may be retried; ensure reprocessing is safe

2. **Set appropriate timeouts** — Use `time_limit` to prevent stuck tasks

3. **Monitor queue depth** — High queue depth indicates workers can't keep up

4. **Use DLQs in production** — Always configure `failed_task_queue_name`

5. **Use managed Redis for production** — Avoid data loss with persistence/replication
